# 02 — Limpeza de Textos

Notebook atualizado para a versão final com base textual completa e amostra rotulada de 3 classes.

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

## 1. Carregar base textual

In [ ]:
base_path = Path('../data/processed/base_textual.csv')
if not base_path.exists():
    base_path = Path('data/processed/base_textual.csv')

df = pd.read_csv(base_path)
print('Registros:', len(df))
print(df.columns.tolist())
display(df.head())

## 2. Funções de limpeza

In [ ]:
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ''
    texto = unicodedata.normalize('NFC', texto)
    texto = re.sub(r'[ 	]+', ' ', texto)
    texto = re.sub(r'
{3,}', '

', texto)
    texto = re.sub(r'^\s*\d+\s*$', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'[ --]', '', texto)
    texto = texto.replace('--', '-')
    return texto.strip()

## 3. Diagnóstico dos textos

In [ ]:
df['texto_limpo'] = df['texto'].apply(limpar_texto)
df['tamanho_texto'] = df['texto_limpo'].str.len()
print(df['tamanho_texto'].describe())
print('
Textos vazios:', (df['tamanho_texto'] == 0).sum())

## 4. Gerar amostra rotulada de 3 classes

A amostra final usa 200 registros por classe, mapeando `tipo_ato` para `rotulo`.

In [ ]:
mapeamento = {'Decretos': 'decreto', 'Leis': 'lei', 'Portarias': 'portaria'}
rot = df[df['tipo_ato'].isin(mapeamento)].copy()
rot['rotulo'] = rot['tipo_ato'].map(mapeamento)
rot['texto'] = rot['texto_limpo']

amostras = []
for classe in ['decreto', 'lei', 'portaria']:
    subset = rot[rot['rotulo'] == classe]
    n = min(200, len(subset))
    amostras.append(subset.sample(n=n, random_state=42))

amostra = pd.concat(amostras).sample(frac=1, random_state=42).reset_index(drop=True)
print(amostra['rotulo'].value_counts())
display(amostra.head())

In [ ]:
out_path = Path('../data/processed/amostra_rotulada.csv')
if not out_path.parent.exists():
    out_path = Path('data/processed/amostra_rotulada.csv')

# Descomente para salvar novamente
# amostra.to_csv(out_path, index=False, encoding='utf-8-sig')
print('Arquivo alvo:', out_path)

## 5. Conclusão

A Etapa 2 gera a base textual e a amostra rotulada usada na preparação NLP da Etapa 3. A versão atual trabalha com 600 exemplos balanceados entre Decreto, Lei e Portaria.